# Run Any HuggingFace Model on TPUs with TorchAX

**A complete, beginner-friendly tutorial**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agemagician/torchax-huggingface/blob/main/notebooks/torchax_huggingface_tutorial.ipynb)

In this notebook, you will learn how to:
1. Run a HuggingFace PyTorch model on JAX with **zero model code changes**
2. Speed up inference **10-100x** using JIT compilation
3. Perform **text classification** and **text generation**
4. Distribute inference across **multiple devices** with tensor parallelism
5. Build a simple **chatbot**

We use [torchax](https://github.com/google/torchax), a library from Google that bridges PyTorch and JAX.

**Model:** `google/gemma-3-1b-it` (1B params, fits on free Colab TPU/GPU)

---

**Blog post:** [Run Any HuggingFace Model on TPUs: A Beginner's Guide to TorchAX](https://dev.to/ahmed_elnaggar/run-any-huggingface-model-on-tpus-a-beginners-guide-to-torchax)

**Credits:** This tutorial builds on the [3-part blog series](https://huggingface.co/blog/qihqi/huggingface-jax-01) by Han Qi ([@qihqi](https://github.com/qihqi)), author of torchax.

## 1. Setup: Detect Runtime & Install Dependencies

This cell auto-detects whether you are running on TPU or GPU and installs the correct JAX variant.

In [ ]:
import subprocess
import sys
import glob
import os

def detect_accelerator():
    """Detect available accelerator: TPU, GPU, or CPU."""
    try:
        import jax
        devices = jax.devices()
        if any(d.platform == "tpu" for d in devices):
            return "tpu"
        if any(d.platform == "gpu" for d in devices):
            return "gpu"
    except Exception:
        pass

    # Fallback: check for TPU device paths (old /dev/accel0 and new /dev/vfio/*)
    if os.path.exists("/dev/accel0") or glob.glob("/dev/vfio/*"):
        return "tpu"
    # Fallback: check for Colab/GCP TPU environment variables
    if os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"):
        return "tpu"

    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True)
        if result.returncode == 0:
            return "gpu"
    except Exception:
        pass

    return "cpu"

accelerator = detect_accelerator()
print(f"Detected accelerator: {accelerator}")

In [ ]:
import os

# Install once, then restart runtime so the fresh packages are loaded
# (Colab pre-installs an older JAX; without a restart, stale modules stay cached in memory)
_marker = "/tmp/.torchax_installed"
if not os.path.exists(_marker):
    # Install PyTorch CPU (torchax handles the accelerator via JAX)
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

    # Install JAX + dependencies in a single pip call
    if accelerator == "tpu":
        !pip install -q -U 'jax[tpu]' torchax transformers flax
    elif accelerator == "gpu":
        !pip install -q -U 'jax[cuda12]' torchax transformers flax
    else:
        !pip install -q -U jax torchax transformers flax

    open(_marker, "w").close()
    print("Restarting runtime to load fresh packages...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("Packages already installed, skipping.")

In [ ]:
# Verify installation
import jax
import torch
import torchax
import transformers

print(f"JAX version:          {jax.__version__}")
print(f"JAX devices:          {jax.devices()}")
print(f"PyTorch version:      {torch.__version__}")
print(f"torchax version:      {torchax.__version__}")
print(f"transformers version: {transformers.__version__}")

## 2. Background: What is TorchAX?

### The Problem
HuggingFace removed native JAX/TensorFlow support from `transformers` in 2024 to focus on PyTorch. This left JAX users (especially on TPUs) without easy access to HuggingFace's model collection.

### The Solution: TorchAX
[torchax](https://github.com/google/torchax) is a library from Google that creates a special `torch.Tensor` subclass wrapping a `jax.Array`. When PyTorch operations run on this tensor, torchax intercepts them and runs the JAX equivalent.

```
PyTorch Model
    |
    v
torchax.Tensor (looks like torch.Tensor, actually holds jax.Array)
    |
    v
JAX execution on TPU/GPU
```

Think of it like a Trojan horse: PyTorch *thinks* it's working with regular tensors, but computation happens on JAX.

### Key Benefits
- **JIT Compilation**: 10-100x speedup via XLA compiler
- **TPU Support**: Native access to Google TPUs
- **Automatic Parallelism**: Distribute across devices with minimal code
- **Zero Model Changes**: Use any `torch.nn.Module` as-is

## 3. Key Concepts

### Pytrees
A **pytree** is a nested structure of containers (dicts, lists, tuples) with arrays as leaves. JAX uses pytrees everywhere. When JAX encounters a custom type (like HuggingFace's `CausalLMOutputWithPast`), it needs to be taught how to unpack and repack it — this is called **pytree registration**.

*Analogy: A pytree is like a shipping box with labeled compartments. JAX knows how to open standard boxes (dicts, lists), but needs instructions for custom ones.*

### JIT Compilation
**JIT** (Just-In-Time) compilation records all operations on the first call, compiles them into an optimized program, and runs the compiled version on subsequent calls.

```
First call:  trace → compile → execute  (slow)
Later calls: execute compiled code       (fast!)
```

*Analogy: Like translating a recipe once into machine instructions vs. reading it word-by-word each time.*

### Static vs. Dynamic Values
During JIT tracing, JAX treats inputs as abstract shapes. If code branches on a value (e.g., `if use_cache:`), that value must be marked **static** — known at compile time.

## 4. Load the Model

We use `google/gemma-3-1b-it` — a 1B parameter instruction-tuned model that fits on free Colab hardware. To use a larger model, change `model_name` to `"google/gemma-3-7b-it"` (needs more memory).

In [ ]:
import torch
import torchax
import jax
import time

from transformers import AutoModelForCausalLM, AutoTokenizer

# Change this to use a different model
MODEL_NAME = "google/gemma-3-1b-it"
# MODEL_NAME = "google/gemma-3-7b-it"  # Larger model (needs Colab Pro or multi-device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cpu"
)

# Enable torchax globally AFTER model loading
# to prevent intercepting unsupported initialization ops (like torchvision's cuda check)
torchax.enable_globally()

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

## 5. Move Model to JAX & Run a Forward Pass (Eager Mode)

In [ ]:
# Move model weights to the JAX device
model.to("jax")

# Tokenize an input prompt
prompt = "The secret to baking a good cake is"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to("jax")

print(f"Input shape: {input_ids.shape}")
print(f"Input tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")

In [ ]:
# Run an eager forward pass
start = time.perf_counter()
with torch.no_grad():
    outputs = model(input_ids, use_cache=False)
eager_time = time.perf_counter() - start

print(f"Output logits shape: {outputs.logits.shape}")
print(f"  (batch_size=1, seq_len={outputs.logits.shape[1]}, vocab_size={outputs.logits.shape[2]})")
print(f"\nEager forward pass took: {eager_time:.3f}s")

# What does the model predict as the next token?
next_token_id = torch.argmax(outputs.logits[0, -1]).item()
print(f"\nPredicted next token: '{tokenizer.decode([next_token_id])}'")

### What just happened?

1. `model.to("jax")` moved every parameter from CPU to the JAX device (like `.to("cuda")` for GPUs)
2. The forward pass ran through PyTorch's code, but every operation was executed by JAX
3. The output `logits` has shape `(1, seq_len, vocab_size)` — a score for every token in the vocabulary at every position

The eager pass works but is slow. Let's speed it up with JIT compilation.

## 6. JIT Compilation: The `extract_jax` Approach

`torchax.extract_jax()` converts a PyTorch model into a pure JAX function.

Before we can JIT it, we need to **register HuggingFace output types** as JAX pytrees — teaching JAX how to unpack and repack these custom types.

In [ ]:
from jax.tree_util import register_pytree_node
from transformers import modeling_outputs, cache_utils

# Register CausalLMOutputWithPast as a pytree
# This tells JAX: "to flatten this object, call .to_tuple(); to reconstruct, call the constructor"
def output_flatten(v):
    return v.to_tuple(), None

def output_unflatten(aux, children):
    return modeling_outputs.CausalLMOutputWithPast(*children)

register_pytree_node(
    modeling_outputs.CausalLMOutputWithPast,
    output_flatten,
    output_unflatten,
)

# Register DynamicCache as a pytree
def _flatten_dynamic_cache(cache):
    return (cache.key_cache, cache.value_cache), None

def _unflatten_dynamic_cache(aux, children):
    c = cache_utils.DynamicCache()
    c.key_cache, c.value_cache = children
    return c

register_pytree_node(
    cache_utils.DynamicCache,
    _flatten_dynamic_cache,
    _unflatten_dynamic_cache,
)

print("Pytree registrations complete.")

In [ ]:
# Extract JAX function and weights
weights, jax_func = torchax.extract_jax(model)

print(f"Number of weight tensors: {len(weights)}")
print(f"Total parameters: {sum(w.size for w in weights.values()) / 1e9:.1f}B")

### Handle Static Arguments

The `use_cache` flag is a boolean. JAX can't trace booleans (it needs their value at compile time). We use a **closure** to bake it in as a constant:

In [ ]:
# Wrap use_cache=False as a closure (compile-time constant)
def forward_no_cache(weights, input_ids):
    return jax_func(weights, (input_ids,), {"use_cache": False})

# JIT compile
jitted_forward = jax.jit(forward_no_cache)

# Convert input to a native JAX array
jax_input_ids = jax.device_put(inputs["input_ids"].numpy())

# Warm-up run (triggers compilation)
print("Compiling (first run)...")
start = time.perf_counter()
res = jitted_forward(weights, jax_input_ids)
jax.block_until_ready(res)
compile_time = time.perf_counter() - start
print(f"First run (compile + execute): {compile_time:.3f}s")

In [ ]:
# Benchmark: Eager vs JIT
print("Benchmarking JIT-compiled forward pass...")
jit_times = []
for i in range(5):
    start = time.perf_counter()
    res = jitted_forward(weights, jax_input_ids)
    jax.block_until_ready(res)
    elapsed = time.perf_counter() - start
    jit_times.append(elapsed)
    print(f"  Run {i}: {elapsed:.4f}s")

avg_jit = sum(jit_times) / len(jit_times)
print(f"\n--- Results ---")
print(f"Eager mode:  {eager_time:.4f}s")
print(f"JIT average: {avg_jit:.4f}s")
print(f"Speedup:     {eager_time / avg_jit:.1f}x")

## 7. The Simpler API: `torchax.compile()`

The `extract_jax` + manual JIT approach gives full control. For most use cases, there is a simpler one-liner:

In [ ]:
import torch.nn as nn

# To avoid JAX trying to trace the boolean use_cache flag, we wrap it in a small module
class NoCacheModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, input_ids):
        # Return only the logits to avoid HuggingFace output class pytree issues
        return self.base_model(input_ids, use_cache=False, return_dict=False)[0]

# One-liner compilation
compiled_model = torchax.compile(NoCacheModel(model))

# Use like a normal PyTorch model
with torch.no_grad():
    logits = compiled_model(input_ids)

print(f"Output shape: {logits.shape}")
next_token = tokenizer.decode([torch.argmax(logits[0, -1]).item()])
print(f"Predicted next token: '{next_token}'")

## 8. Text Classification

Since Gemma is instruction-tuned, we can use prompt engineering for classification tasks:

In [ ]:
def classify_sentiment(text, model, tokenizer):
    """Classify text sentiment using the LLM as a classifier."""
    prompt = f"""Classify the following text as POSITIVE or NEGATIVE.
Text: \"{text}\"
Classification:"""

    inputs = tokenizer(prompt, return_tensors="pt")
    ids = inputs["input_ids"].to("jax")

    with torch.no_grad():
        outputs = model(ids, use_cache=False)

    next_token_logits = outputs.logits[0, -1, :]
    next_token_id = torch.argmax(next_token_logits).item()
    return tokenizer.decode([next_token_id]).strip()

# Test on sample texts
test_texts = [
    "This movie was absolutely fantastic, I loved every minute!",
    "The service was terrible and the food was cold.",
    "A perfectly average experience, nothing special.",
]

print("Sentiment Classification Results:")
print("=" * 60)
for text in test_texts:
    start = time.perf_counter()
    result = classify_sentiment(text, model, tokenizer)
    elapsed = time.perf_counter() - start
    print(f"  [{elapsed:.3f}s] {text[:50]}...")
    print(f"           => {result}")

## 9. Text Generation (Autoregressive Decoding)

### How It Works

An LLM predicts one token at a time:
```
Iteration 1: "The secret is" → predict → " butter"
Iteration 2: "The secret is butter" → predict → " and"
...
```

### The KV Cache

Without caching, the model reprocesses all previous tokens each iteration. The **KV cache** stores intermediate results so only the new token needs processing:

| | Input | Cache | Output |
|---|---|---|---|
| Iter 1 | (1, n) | empty | logits + cache(n) |
| Iter 2 | (1, 1) | cache(n) | logits + cache(n+1) |
| Iter 3 | (1, 1) | cache(n+1) | logits + cache(n+2) |

**DynamicCache**: grows each step (shapes change → bad for JIT)

**StaticCache**: fixed maximum length (shapes stay constant → JIT-friendly)

In [ ]:
from transformers.cache_utils import StaticCache
from jax.tree_util import register_pytree_node
import jax

# Updated registration for StaticCache to handle internal attribute names
def _flatten_static_cache(cache):
    # Accessing the underlying tensors used in StaticCache
    return (
        cache.key_cache, cache.value_cache
    ), (cache.config, cache.max_batch_size, cache.max_cache_len, getattr(cache, "device", None), getattr(cache, "dtype", None))

def _unflatten_static_cache(aux, children):
    config, max_batch_size, max_cache_len, device, dtype = aux
    # Reconstruct the cache object shell
    kwargs = {}
    if device is not None: kwargs["device"] = device
    if dtype is not None: kwargs["dtype"] = dtype
    cache = StaticCache(config, max_batch_size, max_cache_len, **kwargs)
    cache.key_cache, cache.value_cache = children
    return cache

# Clear previous registration if it exists to avoid ValueError
try:
    if hasattr(jax._src.tree_util.default_registry, "custom_dict"):
        jax._src.tree_util.default_registry.custom_dict.pop(StaticCache, None)
except Exception:
    pass

try:
    register_pytree_node(
        StaticCache,
        _flatten_static_cache,
        _unflatten_static_cache,
    )
    print("StaticCache pytree registration updated successfully.")
except ValueError:
    print("StaticCache was already registered. To apply new registration logic, please restart the Colab kernel (Runtime > Restart session) and run the cells again.")

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=50):
    """Generate text using autoregressive decoding with StaticCache."""
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to("jax")
    batch_size, seq_length = input_ids.shape

    # Create a static cache with fixed maximum length
    past_key_values = StaticCache(
        config=model.config,
        max_batch_size=1,
        max_cache_len=seq_length + max_new_tokens,
        device="jax",
        dtype=model.dtype,
    )
    cache_position = torch.arange(seq_length, device="jax")

    # Prefill: process the full prompt
    with torch.no_grad():
        logits, past_key_values = model(
            input_ids,
            cache_position=cache_position,
            past_key_values=past_key_values,
            return_dict=False,
            use_cache=True,
        )

    next_token = torch.argmax(logits[:, -1], dim=-1)[:, None]
    generated_ids = [next_token[:, 0].item()]
    cache_position = torch.tensor([seq_length], device="jax")

    # Decode: generate one token at a time
    for _ in range(max_new_tokens - 1):
        with torch.no_grad():
            logits, past_key_values = model(
                next_token,
                cache_position=cache_position,
                past_key_values=past_key_values,
                return_dict=False,
                use_cache=True,
            )
        next_token = torch.argmax(logits[:, -1], dim=-1)[:, None]
        token_id = next_token[:, 0].item()

        if token_id == tokenizer.eos_token_id:
            break
        generated_ids.append(token_id)
        cache_position += 1

    return tokenizer.decode(generated_ids, skip_special_tokens=True)

# Generate text!
prompt = "The secret to baking a good cake is"
print(f"Prompt: {prompt}")
print(f"\nGenerated:")

start = time.perf_counter()
result = generate_text(model, tokenizer, prompt, max_new_tokens=50)
elapsed = time.perf_counter() - start

print(result)
print(f"\nGeneration time: {elapsed:.2f}s")

## 10. Distributed Inference (Tensor Parallelism)

If you have multiple devices (TPU v2-8, multi-GPU), you can shard model weights for parallel inference.

**How it works:**
- **Column-parallel**: Q, K, V, Gate, Up projections split along output dim
- **Row-parallel**: O, Down projections split along input dim
- JAX's gSPMD inserts all-reduce operations automatically

> **Note:** This section requires a multi-device environment. On single-device Colab, this is for illustration only.

In [ ]:
from jax.sharding import PartitionSpec as P, NamedSharding

num_devices = jax.device_count()
print(f"Available devices: {num_devices}")
print(f"Device types: {[d.platform for d in jax.devices()]}")

if num_devices > 1:
    # Create a device mesh
    mesh = jax.make_mesh((num_devices,), ("axis",))

    def shard_weights(mesh, weights):
        """Shard model weights according to tensor parallelism scheme."""
        sharded = {}
        for name, tensor in weights.items():
            if any(k in name for k in ["q_proj", "k_proj", "v_proj", "gate_proj", "up_proj"]):
                spec = P("axis", None)  # Column-parallel
            elif any(k in name for k in ["o_proj", "down_proj", "lm_head", "embed_tokens"]):
                spec = P(None, "axis")  # Row-parallel
            else:
                spec = P()  # Replicate
            sharded[name] = jax.device_put(tensor, NamedSharding(mesh, spec))
        return sharded

    # Shard the weights
    weights = shard_weights(mesh, weights)

    # Replicate inputs across all devices
    input_ids_sharded = jax.device_put(
        inputs["input_ids"], NamedSharding(mesh, P())
    )

    # Benchmark distributed forward pass
    print("\nBenchmarking distributed forward pass...")
    for i in range(3):
        start = time.perf_counter()
        res = jitted_forward(weights, input_ids_sharded)
        jax.block_until_ready(res)
        elapsed = time.perf_counter() - start
        print(f"  Run {i}: {elapsed:.4f}s")
else:
    print("\nOnly 1 device available — tensor parallelism requires multiple devices.")
    print("To try this, use a TPU v2-8 runtime or multi-GPU setup.")

## 11. Build a Mini Chatbot

Let's wrap everything into a chat function using Gemma's instruction template:

In [ ]:
def chat(user_message, max_new_tokens=100):
    """Send a message and get a response from Gemma."""
    prompt = f"<start_of_turn>user\n{user_message}<end_of_turn>\n<start_of_turn>model\n"
    return generate_text(model, tokenizer, prompt, max_new_tokens)

# Example conversation
questions = [
    "What is JAX and why would I use it instead of PyTorch?",
    "Explain tensor parallelism in simple terms.",
    "Write a haiku about machine learning.",
]

for q in questions:
    print(f"User: {q}")
    response = chat(q)
    print(f"Gemma: {response}")
    print("-" * 60)

## 12. Swapping to a Larger Model

To use a larger model, simply change `MODEL_NAME` at the top of the notebook:

```python
MODEL_NAME = "google/gemma-3-7b-it"  # 7B params
```

The rest of the code remains identical. Other compatible models:
- `google/gemma-3-1b-it` (default, 1B)
- `google/gemma-3-7b-it` (7B, needs more memory)
- `gpt2` (124M, very fast)
- Any `AutoModelForCausalLM`-compatible model

## 13. Troubleshooting

| Error | Cause | Fix |
|---|---|---|
| `TypeError: ... is not a valid JAX type` | Custom type not registered as pytree | Register with `register_pytree_node()` |
| `ConcretizationTypeError` | Boolean/int used in branch during JIT trace | Make it static via closure or `static_argnums` |
| `UserWarning: large constants captured` | Model weights inlined in compiled graph | Pass weights as explicit function argument |
| `RuntimeError: No available devices` | JAX can't see accelerator | Check `jax.devices()`, verify Colab runtime type |
| `OutOfMemoryError` | Model too large for device | Use bfloat16, smaller model, or multi-device |

## 14. Conclusion & Resources

In this tutorial, we:
1. Ran a HuggingFace model on JAX with `model.to("jax")`
2. Achieved massive speedups with JIT compilation
3. Performed text classification and text generation
4. Explored distributed inference with tensor parallelism
5. Built a simple chatbot

### Resources
- [torchax GitHub](https://github.com/google/torchax)
- [torchax Docs](https://google.github.io/torchax/)
- [Original tutorial series by Han Qi](https://huggingface.co/blog/qihqi/huggingface-jax-01)
- [JAX Documentation](https://docs.jax.dev/)
- [HuggingFace LLM Optimization](https://huggingface.co/docs/transformers/llm_optims)
- [Blog post version of this tutorial](https://dev.to/ahmed_elnaggar/run-any-huggingface-model-on-tpus-a-beginners-guide-to-torchax)

### Credits
- **Han Qi ([@qihqi](https://github.com/qihqi))** — author of torchax and the original tutorial series
- **torchax team at Google** — library development
- **HuggingFace** — transformers ecosystem
- **JAX team at Google** — JAX, XLA, and TPU support